In [ ]:
def cal_time_TDI_copy_detect(esn, X_test, n_nodes, n_nodes_dim, Ndim, n_rep, window, target_i, target_j, n_washout, n_evo, A, T, j_threshold = 0.01):
    Error = np.zeros((n_rep, n_nodes, n_nodes))
    for rep in range(n_rep):
        offset = A[rep]
        X_test_washout = X_test[- n_washout + int(offset) : int(offset)]
        Y = X_test[int(offset) : int(offset) + n_evo]
        
        r0 = esn.open_loop(X_test_washout, np.zeros(esn.n_units))[-1]
        Yh = esn.evolve(r0, n_evo)
        
        E = np.zeros((n_nodes, n_nodes))
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i == j:
                    continue
                targetI = n_nodes_dim*i + target_i
                targetJ = n_nodes_dim*j + n_nodes*n_nodes_dim*T + target_j

                jmean = np.mean(Yh[:,targetJ], axis=0)
                if np.abs(jmean) < j_threshold:
                    E[i,j] = -1
                    continue

                Yhji = esn.evolve_edge_removal(r0, targetJ, targetI, n_evo)

                d = (Yh - Yhji)[:,targetI]

                # delta_Yh = Yh[1:, targetI] - Yh[:-1, targetI]
                # delta_Yhji = Yhji[1:, targetI] - Yhji[:-1, targetI]

                # d = delta_Yh - delta_Yhji
                E[i,j] = np.sum(d ** 2) / n_evo / jmean

        Error[rep] = E

    TDI = np.zeros((n_nodes, n_nodes))
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i == j:
                continue
            t = Error[:,i,j]
            e = t[t > 0]
            if e.shape[0] == 0:
                print(f'warning: too large threshold, i:{i},j:{j} no error data')
                TDI[i,j] = 0
            else:
                TDI[i,j] = np.mean(e)
                
    return TDI